# 6 Composition: two representations, and the ablation that keeps you honest

`05_long_documents` used one representation of an opinion: 384 opaque dimensions from a frozen
encoder. This notebook adds a second one — **58 features you can read**, built from regexes
that recognise what a US opinion is made of: citations, statutes, amendments, procedural
posture, doctrine. Then it does what vectormesh exists for: **composes** them.

Two questions drive the notebook:

1. *Where* do you fuse two streams — after each is reduced to a vector per document, or per
   chunk, while the sequence axis still exists?
2. Does any of it help?

The second question has a measured answer, and it is **no** — and that is not a reason to skip
this notebook. The components you meet here (`Parallel`, `Concatenate2D/3D`, `CollateParallel`,
`Gate`, `Highway`, `MoE`) are how real architectures are assembled, and the skill this
notebook actually teaches is the one that separates an experiment from a demo: **change one
variable at a time before believing your own comparison table.** You will watch a conclusion
that looks solid get overturned by exactly that.

In [ ]:
from pathlib import Path
import time
import warnings

import torch
from datasets import load_dataset
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader

from notebooktester import param

warnings.simplefilter("ignore", UserWarning)

ARTEFACTS = Path("../../artefacts").resolve()
SCOTUS_DIR = ARTEFACTS / "scotus_probe"
K, EPOCHS = 13, param(8, test=1)

## 6.1 Stream 1: the embedding cache from the previous notebook

We reuse the cache `05_long_documents` built (or build it here if you skipped ahead — the
build branch is identical) via `cache_utils.get_or_create_cache`, the same load-or-build
helper that notebook uses. One definition, imported by both — the `cnn.py` pattern from
lesson 2 — and it is not only for avoiding copy-paste: `VectorCache.create`'s folder name
carries no trace of which *encoder* built it
([raoulg/vectormesh#1](https://github.com/raoulg/vectormesh/issues/1), open), so a hand-rolled
`glob` here would happily hand you back a stale cache from a different model if you ever
change `MODEL_NAME`. `get_or_create_cache` folds the encoder into the cache tag and raises if
a hit's stored `model_tag` disagrees with the vectorizer you asked for.

In [ ]:
from vectormesh import Vectorizer
from vectormesh.data.vectorizers import detect_device

from cache_utils import get_or_create_cache

MODEL_NAME = "ibm-granite/granite-embedding-small-english-r2"
device = detect_device()


def build_split(name: str):
    n = {"train": 600, "test": 300}[name]
    full = load_dataset("coastalcph/lex_glue", "scotus")[name]
    return full.shuffle(seed=42).select(range(n))


vectorizer = Vectorizer(model_name=MODEL_NAME, col_name="embed", input_col="text",
                        max_length=512, device=device)
embed_train = get_or_create_cache(SCOTUS_DIR, "scotus_train", vectorizer,
                                  lambda: build_split("train"))
embed_test = get_or_create_cache(SCOTUS_DIR, "scotus_test", vectorizer,
                                 lambda: build_split("test"))
assert embed_train.metadata is not None
meta = embed_train.metadata["embed"]
D = meta["hidden_size"]
print({k: v for k, v in meta.items() if k != "chunk_sizes"})

## 6.2 Stream 2: features you design yourself

`legal_patterns.py` (next to this notebook) holds a pattern builder and harmonizer for US
opinions — the same two-function protocol as the `RegexVectorizer` you met in lesson 1, and
the US counterpart of the Dutch legal patterns vectormesh ships for its own `aktes` example.
Skim the module before running on: every feature it produces has a name and a reason.

The class doing the work here is **`ChunkedRegexVectorizer`**. Instead of one feature vector
per document it emits a **`(C, F)` matrix whose row *i* describes chunk *i* of the
embedding**. That alignment is not luck — chunk boundaries are fully determined by the triple
`(tokenizer, context_size, stride)`, so we read all three **from the embedding cache's own
metadata** and pass them in. Never re-type these numbers by hand; §6.4 shows what happens if
they drift.

In [ ]:
from legal_patterns import build_scotus_pattern, harmonize_scotus
from vectormesh import ChunkedRegexVectorizer

regex_vectorizer = ChunkedRegexVectorizer(
    col_name="chunked_regex", input_col="text", device="cpu",
    # the alignment contract: the embedding cache's own settings, not retyped numbers
    tokenizer_name=meta["model_tag"],
    context_size=meta["context_size"],
    stride=meta["stride"],
    pattern_builder=build_scotus_pattern,
    harmonizer=harmonize_scotus,
    training_texts=list(embed_train["text"]),
    min_doc_frequency=5, max_features=200,
)

assert embed_train.dataset is not None
assert embed_test.dataset is not None
traincache = get_or_create_cache(SCOTUS_DIR, "scotus_train", regex_vectorizer, embed_train.dataset)
testcache = get_or_create_cache(SCOTUS_DIR, "scotus_test", regex_vectorizer, embed_test.dataset)
assert traincache.metadata is not None
F = traincache.metadata["chunked_regex"]["hidden_size"]
print(f"{F} fitted features")

Seconds, not minutes — fitting the patterns and writing both matrices costs about as much as
embedding *four documents*. A designed representation is expensive in *your* time and nearly
free in compute; a learned one is the other way around. That trade-off is the designed/learned
line from lesson 1, now with a price tag on each side.

Look at what was fitted before trusting it:

In [ ]:
import numpy as np

# feature i's name lives on the fitted vectorizer, not in the cache — one of the
# things a purely numeric artefact cannot tell you (cache_is_a_contract has limits)
idx_to_name = {i: name for name, i in regex_vectorizer._pattern_to_idx.items()}

# a document "has" feature i if any of its chunks matched it
mat = np.stack([np.asarray(m).max(axis=0) for m in traincache["chunked_regex"]])
doc_freq = mat.mean(axis=0)
order = np.argsort(-doc_freq)
print(f"{'feature':>24} {'in % of docs':>13}")
for i in order[:8]:
    print(f"{idx_to_name[i]:>24} {doc_freq[i]:>12.1%}")
print(f"{'...':>24}")
for i in order[-3:]:
    print(f"{idx_to_name[i]:>24} {doc_freq[i]:>12.1%}")

The most common features (`cite:US`, `cite:SCT`) appear in **every** opinion — of course they
do, every Supreme Court opinion cites Supreme Court cases. A feature present everywhere
carries no information; it is `min_doc_frequency`'s mirror image, and it is the honest reason
this stream will underperform below. Designing features that *discriminate* is harder than
designing features that *match* — exercise E7 is about closing that gap.

## 6.3 The alignment contract, verified

Before fusing anything, check the claim the whole notebook rests on: chunk *i* of the regex
matrix describes chunk *i* of the embedding.

In [ ]:
for name, cache in [("train", traincache), ("test", testcache)]:
    assert cache.metadata is not None
    emb_counts = [len(r) for r in cache["embed"]]
    rgx_counts = [len(r) for r in cache["chunked_regex"]]
    mismatches = sum(a != b for a, b in zip(emb_counts, rgx_counts))
    print(f"{name}: {len(emb_counts)} docs, chunk-count mismatches: {mismatches}, "
          f"offsets_supported: {cache.metadata['chunked_regex']['offsets_supported']}")

## 6.4 The bug that does not crash

Now the reason §6.2 insisted on reading `context_size` and `stride` from the metadata. Build
the regex stream with a *different* window — say 256, because a colleague's config said so —
and try to fuse it anyway:

In [ ]:
from vectormesh.components import FixedPadding

wrong = ChunkedRegexVectorizer(
    col_name="wrong_regex", input_col="text", device="cpu",
    tokenizer_name=meta["model_tag"],
    context_size=256, stride=25,             # <-- NOT the embedding's settings
    pattern_builder=build_scotus_pattern, harmonizer=harmonize_scotus,
    training_texts=list(traincache["text"][:100]), min_doc_frequency=5, max_features=200,
)
sample_texts = list(traincache["text"][:4])
wrong_rows = [torch.tensor(wrong(sample_texts)["wrong_regex"][i]) for i in range(4)]
right_rows = [torch.tensor(traincache[i]["embed"]) for i in range(4)]

pad = FixedPadding(max_chunks=16)
emb_b = pad(right_rows)
rgx_b = pad(wrong_rows)
fused = torch.cat([emb_b, rgx_b], dim=-1)
print(f"embed {tuple(emb_b.shape)} + misaligned regex {tuple(rgx_b.shape)} "
      f"-> fused {tuple(fused.shape)}  — no error raised")
print("raw chunk counts, embed vs misaligned regex:",
      [(e.shape[0], w.shape[0]) for e, w in zip(right_rows, wrong_rows)])

**It runs clean and it is garbage.** A 256-token window produces roughly twice as many chunks,
so row *i* of the regex matrix describes a completely different stretch of text than row *i*
of the embedding — but `FixedPadding` flattens both to 16 rows and `torch.cat` only checks
shapes. Every shape contract in the pipeline is satisfied; the *meaning* is destroyed.

This is the sharpest version of the formal/semantic split (`tensor-ladder.md`) in the whole
course: jaxtyping catches rank and dimension errors, and **nothing mechanical catches semantic
misalignment**. The defence is the contract in §6.2 — build from the cache's own metadata —
plus the verification in §6.3. Neither is optional.

## 6.5 Two streams into one batch

`CollateParallel` is `Collate` for two columns: it yields `((X1, X2), y)` instead of
`(X, y)`. Both padders must use the **same** `max_chunks` — after §6.4 you know why the code
cannot check this for you at the level that matters.

In [ ]:
from vectormesh.data import CollateParallel, OneHot

onehot = OneHot(num_classes=K, label_col="label", target_col="onehot")
train_oh = traincache.map(onehot)
test_oh = testcache.map(onehot)

CMAX = 16
dual = CollateParallel(vec1_col="embed", vec2_col="chunked_regex", target_col="onehot",
                       padder=FixedPadding(max_chunks=CMAX),
                       padder2=FixedPadding(max_chunks=CMAX))

(X1, X2), y = next(iter(DataLoader(train_oh, batch_size=32, collate_fn=dual)))
X1.shape, X2.shape, y.shape

In [ ]:
from vectormesh.components.metrics import Accuracy
from vectormesh.training import Trainer, TrainerSettings

SEEDS = (0, 1, 2)


def to_device(X):
    return tuple(x.to(device) for x in X) if isinstance(X, (tuple, list)) else X.to(device)


def fit_once(make_model, collate, seed: int):
    torch.manual_seed(seed)
    tr = DataLoader(train_oh, batch_size=32, shuffle=True, collate_fn=collate, num_workers=0)
    te = DataLoader(test_oh, batch_size=32, shuffle=False, collate_fn=collate, num_workers=0)
    model = make_model().to(device)

    settings = TrainerSettings(
        epochs=EPOCHS,
        metrics=[Accuracy()],
        logdir=Path("modellogs/composition").resolve(),
        train_steps=len(tr),
        valid_steps=len(te),
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=torch.nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=tr,
        validdataloader=te,
        device=device,
        progress=False,
    )
    trainer.loop()

    model.eval()
    preds, gold = [], []
    with torch.no_grad():
        for X, y in te:
            preds += model(to_device(X)).argmax(-1).cpu().tolist()
            gold += y.argmax(-1).tolist()
    return (sum(p == g for p, g in zip(preds, gold)) / len(gold),
            f1_score(gold, preds, average="macro"),
            sum(p.numel() for p in model.parameters() if p.requires_grad), model)


def fit(make_model, collate):
    # three seeds, as in the previous notebook: the gaps below are a few points,
    # and a few points is exactly what one unlucky init can fake
    runs = [fit_once(make_model, collate, s) for s in SEEDS]
    return {"acc": [r[0] for r in runs], "f1": [r[1] for r in runs],
            "params": runs[0][2], "model": runs[-1][3]}


def show(rows):
    print(f"{'':30} {'accuracy':^17} {'f1_macro':^17} {'params':>10}")
    for name, r in rows.items():
        print(f"{name:30} {sum(r['acc'])/3:.3f} ({min(r['acc']):.3f}-{max(r['acc']):.3f}) "
              f"{sum(r['f1'])/3:.3f} ({min(r['f1']):.3f}-{max(r['f1']):.3f}) "
              f"{r['params']:10,d}")

## 6.6 Fusion altitude 1: after aggregation

Each branch reduces its own stream to `(B, 64)`; `Concatenate2D` glues the two vectors. By
the time the streams meet, the chunk axis is gone — the regex features of chunk 3 can never
interact with the embedding of chunk 3 specifically.

In [ ]:
from vectormesh.components import (Concatenate2D, MeanAggregator, NeuralNet,
                                   Parallel, Serial)


def doc_fusion():
    return Serial([
        Parallel([
            Serial([MeanAggregator(), NeuralNet(hidden_size=D, out_size=64)]),  # (B,C,384)->(B,64)
            Serial([MeanAggregator(), NeuralNet(hidden_size=F, out_size=64)]),  # (B,C,F)  ->(B,64)
        ]),
        Concatenate2D(),                                                        # -> (B,128)
        NeuralNet(hidden_size=128, out_size=K),
    ])


results = {}
results["doc fusion (cat2D)"] = fit(doc_fusion, dual)
show({"doc fusion (cat2D)": results["doc fusion (cat2D)"]})

## 6.7 Fusion altitude 2: per chunk

`Concatenate3D` joins the streams **while the chunk axis still exists**: `(B, C, 384)` and
`(B, C, 58)` become `(B, C, 442)`, so every downstream component sees each chunk's embedding
*next to* that same chunk's legal features. This is the operation §6.3's alignment contract
paid for — and note this pipeline takes the tuple directly, no `Parallel` needed, because
neither branch has private preprocessing.

On IMDB, where nearly every document is one chunk, altitudes 1 and 2 would be the same
operation written twice. Eighteen chunks per document is what makes them different.

In [ ]:
from vectormesh.components import Concatenate3D, Projection

def chunk_fusion(in_dim: int, make_extra=None):
    def build():
        parts = [Concatenate3D(), Projection(hidden_size=in_dim, out_size=64)]
        if make_extra is not None:
            parts.append(make_extra())
        parts += [MeanAggregator(), torch.nn.Dropout(0.2),
                  Projection(hidden_size=64, out_size=K)]
        return Serial(parts)
    return build

results["chunk fusion (cat3D)"] = fit(chunk_fusion(D + F), dual)
show({"chunk fusion (cat3D)": results["chunk fusion (cat3D)"]})

## 6.8 Routing: Gate → Highway → MoE

One idea at increasing resolution, each swapped into the same `Serial` slot:

| component | what it learns |
|---|---|
| `Gate` | a per-dimension scale: `sigmoid(Wx) * x` |
| `Highway` | a two-way interpolation between `x` and `T(x)` |
| `MoE` | an N-way learned mixture: `Σ softmax(Wx)_i · expert_i(x)` |

The mixture-of-experts below routes **per chunk**: each chunk's fused representation gets its
own softmax over two `TransformerBlock` experts. This is (a small version of) the architecture
family behind most current frontier language models — assembled here from parts you have
already met, which is the point of composition.

In [ ]:
from vectormesh.components import MoE, TransformerBlock


def make_moe():
    return MoE(
        experts=[TransformerBlock(hidden_size=64, num_heads=2) for _ in range(2)],
        hidden_size=64, out_size=64,
    )


results["chunk fusion + MoE"] = fit(chunk_fusion(D + F, make_extra=make_moe), dual)
show({"chunk fusion + MoE": results["chunk fusion + MoE"]})

## 6.9 The comparison table — and why you should not believe it yet

Add the single-stream baseline from `05_long_documents` and look at the standings:

In [ ]:
from vectormesh.data import Collate

single = Collate(embedding_col="embed", target_col="onehot",
                 padder=FixedPadding(max_chunks=CMAX))
results["embed only, wide head"] = fit(
    lambda: Serial([MeanAggregator(), NeuralNet(hidden_size=D, out_size=K)]), single)

show(results)

Chunk-level fusion beats the single-stream baseline on accuracy **at a fifth of the
parameters**. Case closed: fusing the regex stream helps?

**No — and this is the most important cell in the notebook.** Compare the winning pipeline to
the baseline again: they differ in *two* ways. One has the regex stream; but one also squeezes
everything through a `Projection` to 64 dimensions before aggregating, while the other feeds
all 384 dimensions to a wide head. Any comparison that changes two variables at once has told
you nothing about either.

## 6.10 The ablation: one variable at a time

Run the *identical* bottleneck pipeline with the regex stream removed. If fusion is what
helped, this should score worse than chunk fusion. If the bottleneck is what helped, it
should score the same or better.

In [ ]:
def bottleneck_single(in_dim: int, make_extra=None):
    def build():
        parts = [Projection(hidden_size=in_dim, out_size=64)]
        if make_extra is not None:
            parts.append(make_extra())
        parts += [MeanAggregator(), torch.nn.Dropout(0.2),
                  Projection(hidden_size=64, out_size=K)]
        return Serial(parts)
    return build

results["ABLATION: same shape, no regex"] = fit(bottleneck_single(D), single)
results["ABLATION: + MoE, no regex"] = fit(bottleneck_single(D, make_extra=make_moe), single)

show(results)

## 6.11 What actually happened

Measured over three seeds while planning this lesson (single seeds here will wobble but the
ranking holds):

- **The regex stream contributes nothing.** The embed-only bottleneck (~0.62 accuracy) beats
  the fused version (~0.61). Adding 58 features that mostly say "this opinion cites cases" —
  which every opinion does — adds parameters, not information.
- **The MoE contributes nothing either.** With or without the regex stream, adding it lands
  within seed noise of leaving it out. Two transformer experts and a learned router, at this
  data budget, are an expensive way to compute an average.
- **What produced the "win" was the bottleneck** — a *smaller* model (25k parameters against
  152k) regularising better on 600 documents. The apparent fusion victory was a parameter
  effect wearing a fusion costume.

This is the fourth time the course has measured an expensive component failing to earn its
place — the flowers CNN (L2), the GRU forecaster (L3), the learned aggregators (previous
notebook), and now fusion + MoE. The pattern is not "fancy things never work". It is:
**a comparison is evidence only when one variable changed**, and the components that survive
that test are rarer than the ones that survive a demo.

And one more thing the table is quietly telling you — look at the two metrics. The bottleneck
has the best **accuracy**; the wide head has the best **macro-F1**. Compressing to 64
dimensions helps the common classes and hurts the rare ones (this dataset has a class with
*three* training examples). Which model is "best" depends on who you are answering to — which
is precisely where the metrics notebook picks up.

## Exercises

**E6 — does the router route?** The trained MoE pipeline is
`results["chunk fusion + MoE"]["model"]`; its `MoE` layer exposes the routing linear as
`.router` (read the source with `MoE??`). Run the test set through it, collect the router's
softmax per document, and plot the expert weights grouped by issue area. Does an expert own criminal procedure, or is every document a 50/50 blend? Then
the parameter-matched question that settles whether the structure earned anything: train
**2 experts of width 64 vs 1 expert of width 128** — same budget, different shape. A collapsed
router is a finding, not a failure: "this MoE is an expensive ensemble average" is exactly the
sentence a working ML engineer needs to be able to say.

**E7 — beat the embedding with features you wrote yourself.** The designed stream lost, and
§6.2 diagnosed why: its most common features appear in 83-100% of documents and carry nothing.
This exercise is open — the course does not know the answer. Edit `legal_patterns.py`: drop
the universal citations, expand the doctrine vocabulary for one issue area you pick, raise
`min_doc_frequency`. How close can 58 *readable* dimensions get to 384 opaque ones — and what
can you say about your model afterwards that the embedding user can never say about theirs?

**E8 — the misalignment, quantified.** §6.4 showed the misaligned fusion runs clean. Finish
the experiment: build the full wrong-window regex cache and train the chunk-fusion pipeline on
it. How much does silent garbage actually cost, and would you have noticed without the
baseline?